# Latent Channel Propagation — Phase 1 Pipeline (Colab orchestrator)

Runs the five Phase 1 data-collection scripts in order:
`01_build_corpus.py` → `02_build_prompt_templates.py` → `03_run_inference_and_cache.py` → `04_label_vea.py` → `05_partition_cells.py`

See `README.md` in this folder for what each script does. This notebook is a thin orchestrator — it does not reimplement pipeline logic, it just calls the scripts with the right arguments and shows you the intermediate outputs.

**Before running:** Runtime → Change runtime type → select a GPU (T4 is fine for the 8B pilot).

## 0. Get the code onto this Colab instance

Pick ONE of the two cells below depending on whether `latent-channel-propagation` is a git repo you've pushed somewhere, or you're just uploading the folder directly.

In [ ]:
# Option A -- clone from your own git remote (uncomment and edit)
!git clone https://github.com/manchitro/latent-channel-propagation latent-channel-propagation
%cd latent-channel-propagation

In [ ]:
# Option B -- mount Google Drive if you've synced the folder there,
# or use the Colab file-upload UI (folder icon in the left sidebar) to
# drag the 01..05 .py files + README.md into /content/latent-channel-propagation
# from google.colab import drive
# drive.mount('/content/drive')

# Example if synced under My Drive/latent-channel-propagation -- edit path as needed:
# %cd /content/drive/MyDrive/latent-channel-propagation

In [ ]:
import os
assert os.path.exists('01_build_corpus.py'), (
    "Not in the right directory -- cd into the folder containing the "
    "01..05 scripts before continuing (see Option A/B above)."
)
print('Working directory OK:', os.getcwd())
!ls

## 1. Install dependencies

In [ ]:
!pip install -q datasets transformers accelerate bitsandbytes huggingface_hub pandas pyarrow

## 2. Authenticate with Hugging Face

Needed for `01_build_corpus.py` (AdvBench + WMDP) and `03_run_inference_and_cache.py` (Llama-3-8B-Instruct is a gated model -- request access on its model page first, then log in below).

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Configuration

`SMOKE_TEST=True` caps the pipeline to a handful of prompts so you can validate the full chain runs end-to-end before committing GPU time to the full corpus. Flip to `False` for the real pilot run.

In [ ]:
SMOKE_TEST = True
SMOKE_LIMIT = 6  # rows fed to 03_run_inference_and_cache.py when SMOKE_TEST

WMDP_CAP = 5 if SMOKE_TEST else 200  # per-subset cap, passed to 01_build_corpus.py

MODEL_ID = "meta-llama/Llama-3-8B-Instruct"  # swap for the 70B scale-up on AIUB hardware
USE_LLM_JUDGE = False  # flip once you've implemented llm_judge_label() in 04_label_vea.py

print(f"SMOKE_TEST={SMOKE_TEST}  WMDP_CAP={WMDP_CAP}  MODEL_ID={MODEL_ID}")

## 4. Step 1 -- Build corpus (AdvBench + WMDP)

In [ ]:
!python 01_build_corpus.py --out corpus.parquet --wmdp-per-subset-cap {WMDP_CAP}

In [ ]:
import pandas as pd
corpus = pd.read_parquet('corpus.parquet')
print(corpus.shape)
corpus.head()

## 5. Step 2 -- Build train/deploy prompt templates

In [ ]:
!python 02_build_prompt_templates.py --corpus corpus.parquet --out templated_corpus.parquet

In [ ]:
templated = pd.read_parquet('templated_corpus.parquet')
print(templated.shape)
templated[['variant_id', 'framing', 'user_prompt']].head()

## 6. Step 3 -- Run inference + cache activations

This is the GPU-heavy step. With `SMOKE_TEST=True` it only runs `SMOKE_LIMIT` rows -- check the printed compliance labels look sane before running the full corpus.

In [ ]:
limit_flag = f"--limit {SMOKE_LIMIT}" if SMOKE_TEST else ""
!python 03_run_inference_and_cache.py \
    --templated templated_corpus.parquet \
    --model-id {MODEL_ID} \
    --four-bit \
    {limit_flag} \
    --out-transcripts transcripts.parquet \
    --out-activations activations.npz

In [ ]:
transcripts = pd.read_parquet('transcripts.parquet')
print(transcripts.shape)
print(transcripts['compliance_label'].value_counts())
transcripts[['variant_id', 'framing', 'compliance_label', 'answer']].head()

In [ ]:
import numpy as np
acts = np.load('activations.npz')
print(f"{len(acts.files)} cached activation vectors")
sample_key = acts.files[0]
print(f"example '{sample_key}' shape: {acts[sample_key].shape}  (num_layers+1, hidden_dim)")

## 7. Step 4 -- Label verbalized evaluation awareness (VEA)

In [ ]:
judge_flag = "--use-llm-judge" if USE_LLM_JUDGE else ""
!python 04_label_vea.py \
    --transcripts transcripts.parquet \
    --out labeled_transcripts.parquet \
    {judge_flag}

In [ ]:
labeled = pd.read_parquet('labeled_transcripts.parquet')
print(labeled['vea_label'].value_counts())
labeled[['variant_id', 'framing', 'vea_label', 'reasoning']].head()

## 8. Step 5 -- Partition into the four analysis cells

In [ ]:
!python 05_partition_cells.py --labeled labeled_transcripts.parquet --out partitioned_corpus.parquet

In [ ]:
partitioned = pd.read_parquet('partitioned_corpus.parquet')
print(partitioned['cell'].value_counts())
partitioned.head()

## 9. Persist outputs

Copy the parquet/npz artifacts to Drive (or download them) before the Colab runtime recycles -- everything under `/content` is ephemeral.

In [ ]:
import shutil, os

DRIVE_OUT_DIR = '/content/drive/MyDrive/latent-channel-propagation/phase1_outputs'
if os.path.exists('/content/drive'):
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
    for f in ['corpus.parquet', 'templated_corpus.parquet', 'transcripts.parquet',
              'activations.npz', 'labeled_transcripts.parquet', 'partitioned_corpus.parquet']:
        if os.path.exists(f):
            shutil.copy(f, DRIVE_OUT_DIR)
    print(f"Copied outputs to {DRIVE_OUT_DIR}")
else:
    print("Drive not mounted -- use the Colab file browser to manually download the outputs.")

## Next steps

- If `SMOKE_TEST` passed and the compliance/VEA labels look reasonable, set `SMOKE_TEST=False` and rerun from Step 1 for the full pilot corpus.
- Once you're happy with the 8B pilot results, swap `MODEL_ID` to the 70B checkpoint and move this notebook (or the equivalent scripts) to the AIUB DGX Spark lab for the scale-up run.
- `partitioned_corpus.parquet` and `activations.npz` are the inputs to Phase 2 (probe training) -- see `Latent Channel Propagation - Feasibility and Impact.md` in the Obsidian vault for the full plan.